# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NimaWyd/Flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**Two signals I'm checking before locking in the rule:**

**Signal 1 — `avg_ctr_march` (links to FlyRank's `low_ctr_visible_page` flag)**
The starter pipeline fires `low_ctr_visible_page` when `impressions >= 500` and `ctr < 0.5` at position 1–20. The claim here is the same: a page getting impressions but not clicks is underperforming for its visibility level, and that pattern tends to accompany or precede decline. I'll bucket CTR into quartiles and check the decline rate per bucket. If lower CTR → higher decline rate: **CONFIRMED**. If the buckets are flat: **FALSE**.

**Signal 2 — `avg_position_march` (links to FlyRank's `page_one_decay_risk` flag)**
The starter fires `page_one_decay_risk` for pages at position ≤ 10 with content age ≥ 180 days. Without content age from the warehouse dim_content, I'm checking the position signal alone: do pages with a stronger position (lower number = closer to rank 1) have a lower decline rate? Or are top-ranked pages declining at the same rate as lower-ranked ones? If top positions protect against decline: **OPPOSITE** (strong position → lower decline). If no pattern: **MIXED** or **FALSE**.

*Verdicts printed by the code cell below — review them before the rule is built.*

---

**Provisional rule (to finalise after verdicts):**
"A page is worth reviewing if it is getting above-median impressions but below-median CTR — visible but not clicking, which is the CTR-fix opportunity the `low_ctr_visible_page` flag was designed for."

Score = `avg_impressions_march × low_ctr_flag`
Reason codes: `high_vol_low_ctr` → `ctr_fix` | `high_vol_ok_ctr` → `monitor` | `low_vol` → `deprioritize`

In [ ]:
# ── SIGNAL 1: avg_ctr_march → decline rate (CTR-fix flag) ─────────────────
# Label used only for evaluation here — not a rule input.
sig1 = merged[['avg_ctr_march','is_declining']].dropna().copy()
sig1['ctr_bucket'] = pd.qcut(sig1['avg_ctr_march'], q=4,
                              labels=['q1_low_ctr','q2','q3','q4_high_ctr'])

ctr_table = (sig1.groupby('ctr_bucket', observed=True)
             .agg(n=('is_declining','count'),
                  decline_rate=('is_declining','mean'))
             .round(3))
print("=== SIGNAL 1: avg_ctr_march by quartile ===")
print(ctr_table.to_string())
print(f"(base rate across all rows: {merged['is_declining'].mean():.3f})")

q1_ctr_rate = ctr_table.loc['q1_low_ctr', 'decline_rate']
q4_ctr_rate = ctr_table.loc['q4_high_ctr', 'decline_rate']
diff1 = q1_ctr_rate - q4_ctr_rate  # positive = lower CTR → higher decline

if abs(diff1) < 0.03:
    v1 = "FALSE"
elif diff1 >= 0.07:
    v1 = "CONFIRMED"   # lower CTR → more decline
elif diff1 <= -0.07:
    v1 = "OPPOSITE"    # lower CTR → less decline
else:
    v1 = "MIXED"

print(f"\nVerdict: {v1}")
print(f"  q1 (lowest CTR) decline rate : {q1_ctr_rate:.3f}")
print(f"  q4 (highest CTR) decline rate: {q4_ctr_rate:.3f}  (diff = {diff1:+.3f})")

# ── SIGNAL 2: avg_position_march → decline rate (page_one_decay_risk flag) ─
sig2 = merged[['avg_position_march','is_declining']].dropna().copy()
sig2['pos_bucket'] = pd.qcut(sig2['avg_position_march'], q=4,
                              labels=['q1_best_pos','q2','q3','q4_worst_pos'])

pos_table = (sig2.groupby('pos_bucket', observed=True)
             .agg(n=('is_declining','count'),
                  decline_rate=('is_declining','mean'))
             .round(3))
print("\n=== SIGNAL 2: avg_position_march by quartile ===")
print("(lower position number = closer to rank 1 = better ranking)")
print(pos_table.to_string())
print(f"(base rate across all rows: {merged['is_declining'].mean():.3f})")

q1_pos_rate = pos_table.loc['q1_best_pos', 'decline_rate']
q4_pos_rate = pos_table.loc['q4_worst_pos', 'decline_rate']
diff2 = q1_pos_rate - q4_pos_rate  # positive = best position → more decline (counterintuitive)

if abs(diff2) < 0.03:
    v2 = "FALSE"
elif diff2 >= 0.07:
    v2 = "CONFIRMED"   # top-ranked pages decline MORE (counterintuitive but noted in ML-02)
elif diff2 <= -0.07:
    v2 = "OPPOSITE"    # top-ranked pages decline less
else:
    v2 = "MIXED"

print(f"\nVerdict: {v2}")
print(f"  q1 (best position) decline rate : {q1_pos_rate:.3f}")
print(f"  q4 (worst position) decline rate: {q4_pos_rate:.3f}  (diff = {diff2:+.3f})")

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.